In [1]:
import pandas as pd
df = pd.read_csv('../data/train.csv')

In [2]:
df.head()

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,0,42,technician,married,secondary,no,7,no,no,cellular,25,aug,117,3,-1,0,unknown,0
1,1,38,blue-collar,married,secondary,no,514,no,no,unknown,18,jun,185,1,-1,0,unknown,0
2,2,36,blue-collar,married,secondary,no,602,yes,no,unknown,14,may,111,2,-1,0,unknown,0
3,3,27,student,single,secondary,no,34,yes,no,unknown,28,may,10,2,-1,0,unknown,0
4,4,26,technician,married,secondary,no,889,yes,no,cellular,3,feb,902,1,-1,0,unknown,1


In [3]:
df = df.iloc[:, 1:] # dropping the id column

In [4]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,42,technician,married,secondary,no,7,no,no,cellular,25,aug,117,3,-1,0,unknown,0
1,38,blue-collar,married,secondary,no,514,no,no,unknown,18,jun,185,1,-1,0,unknown,0
2,36,blue-collar,married,secondary,no,602,yes,no,unknown,14,may,111,2,-1,0,unknown,0
3,27,student,single,secondary,no,34,yes,no,unknown,28,may,10,2,-1,0,unknown,0
4,26,technician,married,secondary,no,889,yes,no,cellular,3,feb,902,1,-1,0,unknown,1


In [5]:
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
# encoding
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# evaluation
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# plotting
import matplotlib.pyplot as plt
import seaborn as sns


In [12]:
# 1. Handle the missing values:- 
# - Drop, Impute (Mean, median, mode, Custering fill ...)
# - Our data doesn't have any missing values.

# 1.1 Handle the duplicates:- 
# - We don't have any duplicates, otherwise we needed to drop.

In [ ]:
# 2. Outliers (for numerical columns)
for col in numerical_cols:
    q1 = df.col.quantile(0.25)
    q3 = df.col.quantile(0.75)
    iqr = q3 - q1
    

In [23]:
q1 - 1.5*iqr, q3+1.5*iqr

(-2085.0, 3475.0)

In [25]:
df.balance.max(), df.balance.min()

(99717, -8019)

In [29]:
len(df[(df.balance > 3475) | (df.balance < -2085)])/len(df)*100

7.699333333333333

In [ ]:
# ML Classification Pipeline for Bank Term Deposit Prediction

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

# 1. DATA PREPROCESSING
# Handle missing values
df.isnull().sum()
df = df.dropna()  # or df.fillna(method='most_frequent')

# Remove duplicates
df = df.drop_duplicates()

# Handle outliers (example for numerical columns)
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return df[(df[column] >= lower) & (df[column] <= upper)]

# Apply to numerical columns
for col in ['age', 'balance', 'duration', 'campaign', 'previous']:
    df = remove_outliers(df, col)

# 2. FEATURE ENGINEERING
# Create new features
df['balance_per_age'] = df['balance'] / df['age']
df['campaign_per_previous'] = df['campaign'] / (df['previous'] + 1)
df['has_previous_contact'] = (df['pdays'] != -1).astype(int)
df['duration_minutes'] = df['duration'] / 60

# Encode categorical variables
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 
                   'contact', 'month', 'poutcome']

# Label Encoding for binary categories
le = LabelEncoder()
df['y'] = le.fit_transform(df['y'])

# One-hot encoding for multi-class categories
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# 3. FEATURE SELECTION
# Correlation analysis
correlation = df_encoded.corr()['y'].abs().sort_values(ascending=False)
print(correlation.head(10))

# Select features based on correlation threshold
selected_features = correlation[correlation > 0.1].index.tolist()
selected_features.remove('y')

# Recursive Feature Elimination
from sklearn.feature_selection import RFE
rf = RandomForestClassifier(random_state=42)
rfe = RFE(rf, n_features_to_select=15)
X_rfe = rfe.fit_transform(df_encoded.drop('y', axis=1), df_encoded['y'])

# 4. DATA SPLITTING
X = df_encoded[selected_features]
y = df_encoded['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                    random_state=42, stratify=y)

# 5. FEATURE SCALING
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. MODEL BUILDING & TRAINING
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'SVM': SVC(random_state=42, probability=True)
}

# Train and evaluate models
results = {}
for name, model in models.items():
    if name in ['Logistic Regression', 'SVM']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
    
    accuracy = model.score(X_test if name not in ['Logistic Regression', 'SVM'] 
                          else X_test_scaled, y_test)
    auc = roc_auc_score(y_test, y_prob)
    
    results[name] = {'accuracy': accuracy, 'auc': auc}
    print(f"{name}: Accuracy={accuracy:.3f}, AUC={auc:.3f}")

# 7. HYPERPARAMETER TUNING
# Example for Random Forest
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10]
}

rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), 
                       param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
rf_grid.fit(X_train, y_train)

best_rf = rf_grid.best_estimator_
print(f"Best RF parameters: {rf_grid.best_params_}")

# 8. MODEL EVALUATION
# Best model predictions
y_pred_best = best_rf.predict(X_test)
y_prob_best = best_rf.predict_proba(X_test)[:, 1]

# Classification report
print(classification_report(y_test, y_pred_best))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

# ROC Curve
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob_best)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc_score(y_test, y_prob_best):.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': best_rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=feature_importance.head(15), x='importance', y='feature')
plt.title('Top 15 Feature Importance')
plt.show()

# 9. CROSS-VALIDATION
cv_scores = cross_val_score(best_rf, X, y, cv=5, scoring='roc_auc')
print(f"Cross-validation AUC: {cv_scores.mean():.3f} (+/- {cv_scores.std() * 2:.3f})")

# 10. ADVANCED TECHNIQUES (Optional)

# Class imbalance handling
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

# Ensemble methods
from sklearn.ensemble import VotingClassifier
voting_clf = VotingClassifier(
    estimators=[('rf', best_rf), ('gb', GradientBoostingClassifier())],
    voting='soft'
)
voting_clf.fit(X_train, y_train)

# Stacking
from sklearn.ensemble import StackingClassifier
stacking_clf = StackingClassifier(
    estimators=[('rf', RandomForestClassifier()), 
                ('gb', GradientBoostingClassifier())],
    final_estimator=LogisticRegression()
)
stacking_clf.fit(X_train, y_train)

# 11. MODEL INTERPRETATION
# SHAP values (install with: pip install shap)
import shap
explainer = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values[1], X_test)

# 12. FINAL MODEL SAVING
import joblib
joblib.dump(best_rf, 'best_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

# Model loading
# loaded_model = joblib.load('best_model.pkl')
# loaded_scaler = joblib.load('scaler.pkl')